In [ ]:
# state = init_agent()
init_agent = lambda *args, **kwargs: None

# state = reset_agent()
reset_agent = lambda *args, **kwargs: None

# action = get_action(state, observation, act_key)
get_action = lambda *args, **kwargs: None

# state = update(state, observation, reward, terminated, truncated, info)
update = lambda *args, **kwargs: None

In [ ]:
import gymnasium as gym
import jax
import jax.numpy as jnp

env = gym.make(
    "CartPole-v1",
    render_mode="none",
)

N_TRIALS = 100
N_REPS = 10
# TODO load other hyperparams in here somehow?

# prng key
key = jax.random.PRNGKey(0)

all_trial_results = []

for rep in range(N_REPS):
    print(f"REPEAT {rep}")

    # independent key per rep, derived from rep index via fold_in
    key = jax.random.fold_in(jax.random.PRNGKey(0), rep)

    # TODO
    state = init_agent()

    trial_results = []

    for trial_idx in range(N_TRIALS):
        print(f"trial {trial_idx}")

        observation, info = env.reset()

        # TODO optionally reset the agent
        state = reset_agent()

        episode_over = False
        steps = 0

        while not episode_over:
            # this is the nice jax way to get a new random seed which is properly independent
            key, act_key = jax.random.split(key)

            # TODO get action (use act_key if it is random)
            action = get_action(state, observation, act_key)

            # execute action in env
            observation, reward, terminated, truncated, info = env.step(action)
            steps += 1

            # TODO do something with the new info
            state = update(state, observation, reward, terminated, truncated, info)

            episode_over = terminated or truncated

        trial_results.append(steps)
        print(f"stayed alive for {steps} steps\n")
    all_trial_results.append(trial_results)

env.close()

all_trial_results = np.array(all_trial_results)  # shape (N_REPS, N_TRIALS)
mean_curve = all_trial_results.mean(axis=0)
std_curve = all_trial_results.std(axis=0)